# Umubyeyi — Colab LoRA fine-tuning

This notebook reproduces the project generator training on a Colab GPU. It uses only the existing source-attributed `postpartum_wellbeing.json` knowledge bank—no additional dataset—and creates a leakage-controlled topic split. The exported adapter is directly compatible with `models/umubyeyi-mt5-lora/` in the app.

Before running: choose **Runtime → Change runtime type → T4 GPU**.

In [ ]:
!pip -q install 'transformers==4.46.3' 'datasets==3.1.0' 'peft==0.13.2' 'accelerate>=1.1' sentencepiece rouge-score

import json, random, re, shutil, time
from pathlib import Path
import torch
from google.colab import files
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE — enable a GPU runtime')
assert torch.cuda.is_available(), 'Enable a Colab GPU runtime before training.'

Upload the project file `data/knowledge/postpartum_wellbeing.json`. This is the same evidence bank used by the application.

In [ ]:
uploaded = files.upload()
assert 'postpartum_wellbeing.json' in uploaded, 'Upload postpartum_wellbeing.json with its original filename.'
bank = json.loads(uploaded['postpartum_wellbeing.json'].decode('utf-8'))
assert len(bank) == 14 and all(row.get('text_en') and row.get('text_rw') for row in bank)
print('Loaded', len(bank), 'source-attributed topics')

In [ ]:
SEED = 42
BASE_MODEL = 'google/mt5-small'
OUTPUT_DIR = Path('/content/umubyeyi-mt5-lora')
PROMPTS = {
 'en': ['I need emotional support with {terms}.', 'After giving birth, I have been struggling with {terms}.', 'Can you help me understand {terms}?', 'I am a new mother dealing with {terms}.', 'What can I do when I experience {terms}?', 'Please support me with {terms}.'],
 'rw': ['Nkeneye ubufasha ku bijyanye na {terms}.', 'Nyuma yo kubyara ndimo guhangana na {terms}.', 'Wamfasha gusobanukirwa {terms}?', 'Ndi umubyeyi mushya mpanganye na {terms}.', 'Nakora iki iyo mfite {terms}?', 'Mfasha ku bijyanye na {terms}.']
}
ACK = {'en': ['Thank you for sharing this.', 'I hear that this is difficult.', 'You are not alone in facing this.'], 'rw': ['Urakoze kubivuga.', 'Ndumva ko ibi bikugoye.', 'Nturi wenyine muri ibi.']}

def format_input(query, evidence, lang):
    language = 'Kinyarwanda' if lang == 'rw' else 'English'
    return ('Generate a brief, empathetic postpartum emotional-support answer. Use only the evidence and answer in the requested language. Do not diagnose.\n' + f'Language: {language}\nEvidence: {evidence.strip()}\nMother: {query.strip()}\nAnswer:')

topic_ids = sorted(row['id'] for row in bank)
random.Random(SEED).shuffle(topic_ids)
split = {topic: ('test' if i < 2 else 'validation' if i < 4 else 'train') for i, topic in enumerate(topic_ids)}
examples = []
for row in bank:
    for lang in ('en', 'rw'):
        evidence, terms = row[f'text_{lang}'].strip(), row[f'queries_{lang}'].strip()
        for variant, template in enumerate(PROMPTS[lang]):
            query = template.format(terms=terms)
            examples.append({'topic_id': row['id'], 'topic': row['topic'], 'language': lang, 'split': split[row['id']], 'input': format_input(query, evidence, lang), 'target': f"{ACK[lang][variant % 3]} {evidence}", 'evidence': evidence})

for name in ('train', 'validation', 'test'):
    rows = [x for x in examples if x['split'] == name]
    print(name, len(rows), 'examples; topics:', sorted({x['topic_id'] for x in rows}))
assert len(examples) == 168
assert not ({x['topic_id'] for x in examples if x['split'] == 'train'} & {x['topic_id'] for x in examples if x['split'] == 'test'})

In [ ]:
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq, EarlyStoppingCallback, Seq2SeqTrainer, Seq2SeqTrainingArguments

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
def tokenize(batch):
    encoded = tokenizer(batch['input'], max_length=512, truncation=True)
    encoded['labels'] = tokenizer(text_target=batch['target'], max_length=220, truncation=True)['input_ids']
    return encoded

datasets = {}
for name in ('train', 'validation'):
    rows = [{'input': x['input'], 'target': x['target']} for x in examples if x['split'] == name]
    datasets[name] = Dataset.from_list(rows).map(tokenize, batched=True, remove_columns=['input', 'target'])

base = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
lora = LoraConfig(task_type=TaskType.SEQ_2_SEQ_LM, r=8, lora_alpha=16, lora_dropout=0.05, target_modules=['q', 'v'], bias='none')
model = get_peft_model(base, lora)
trainable, total = model.get_nb_trainable_parameters()
print(f'Trainable: {trainable:,}/{total:,} ({100*trainable/total:.4f}%)')
test_rows = [x for x in examples if x['split'] == 'test']
def generate_all(current_model):
    predictions = []
    current_model.eval()
    for row in test_rows:
        batch = tokenizer(row['input'], return_tensors='pt', truncation=True, max_length=512).to(current_model.device)
        with torch.inference_mode():
            output = current_model.generate(**batch, max_new_tokens=180, num_beams=2, no_repeat_ngram_size=3)
        predictions.append(tokenizer.decode(output[0], skip_special_tokens=True).strip())
    return predictions

print('Generating untouched-topic baseline before weight updates...')
baseline_predictions = generate_all(model)

args = Seq2SeqTrainingArguments(output_dir='/content/checkpoints', num_train_epochs=12, learning_rate=1e-3, per_device_train_batch_size=2, per_device_eval_batch_size=2, gradient_accumulation_steps=4, eval_strategy='epoch', save_strategy='epoch', logging_steps=5, save_total_limit=2, load_best_model_at_end=True, metric_for_best_model='eval_loss', greater_is_better=False, report_to='none', fp16=True, seed=SEED)
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=datasets['train'], eval_dataset=datasets['validation'], data_collator=DataCollatorForSeq2Seq(tokenizer, model=model), processing_class=tokenizer, callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
started = time.time()
result = trainer.train()
print('Training minutes:', round((time.time()-started)/60, 2))

In [ ]:
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)

def overlap(answer, evidence):
    words = re.findall(r'[^\W\d_]{3,}', answer.lower(), flags=re.UNICODE)
    evidence_words = set(re.findall(r'[^\W\d_]{3,}', evidence.lower(), flags=re.UNICODE))
    return sum(word in evidence_words for word in words) / len(words) if words else 0.0

def strict_accept(answer, evidence):
    normal = lambda text: ' '.join(re.findall(r'[^\W_]+', text.lower(), flags=re.UNICODE))
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', evidence) if s.strip()]
    matched = [s for s in sentences if normal(s) in normal(answer)]
    return sum(len(normal(s).split()) for s in matched) / max(1, sum(len(normal(s).split()) for s in sentences)) >= .65

def metrics_for(predictions):
    rouge = sum(scorer.score(row['target'], pred)['rougeL'].fmeasure for row, pred in zip(test_rows, predictions)) / len(test_rows)
    grounding = sum(overlap(pred, row['evidence']) for row, pred in zip(test_rows, predictions)) / len(test_rows)
    acceptance = {lang: sum(strict_accept(pred, row['evidence']) for row, pred in zip(test_rows, predictions) if row['language'] == lang) / sum(row['language'] == lang for row in test_rows) for lang in ('en', 'rw')}
    return {'rouge_l_f1': round(rouge, 4), 'mean_grounding_overlap': round(grounding, 4), 'strict_acceptance_by_language': {k: round(v, 4) for k,v in acceptance.items()}, 'examples': len(test_rows)}

predictions = generate_all(model)
baseline_metrics = metrics_for(baseline_predictions)
metrics = metrics_for(predictions)
acceptance = metrics['strict_acceptance_by_language']
print('Baseline:', json.dumps(baseline_metrics, indent=2))
print('Fine-tuned:', json.dumps(metrics, indent=2))
print('Review every generated Kinyarwanda sample before enabling that adapter language.')

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
manifest = {'fine_tuned': True, 'method': 'LoRA supervised fine-tuning (PEFT) on Google Colab GPU', 'base_model': BASE_MODEL, 'task': 'bilingual evidence-conditioned postpartum support generation', 'seed': SEED, 'dataset': {'examples': len(examples), 'topics': len(bank), 'splits': {s: sum(x['split']==s for x in examples) for s in ('train','validation','test')}, 'languages': {lang: sum(x['language']==lang for x in examples) for lang in ('en','rw')}}, 'data_statement': 'Existing source-attributed project knowledge only; no external conversational dataset.', 'accepted_generation_languages': [lang for lang, rate in acceptance.items() if rate >= .5], 'trainable_parameters': trainable, 'total_parameters': total, 'trainable_percentage': round(100*trainable/total, 4), 'epochs_requested': 12, 'training_loss': round(float(result.training_loss), 6), 'baseline_test': baseline_metrics, 'fine_tuned_test': metrics}
(OUTPUT_DIR/'training_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
(OUTPUT_DIR/'test_generations.json').write_text(json.dumps([{**{k: row[k] for k in ('topic','language','evidence','target')}, 'prediction': pred} for row,pred in zip(test_rows,predictions)], ensure_ascii=False, indent=2), encoding='utf-8')
archive = shutil.make_archive('/content/umubyeyi-mt5-lora', 'zip', OUTPUT_DIR)
print('Saved:', archive)
files.download(archive)

After downloading, extract the archive into `models/umubyeyi-mt5-lora/`, preserving `training_manifest.json`, the tokenizer files, and `adapter_model.safetensors`. Do not enable Kinyarwanda in the adapter manifest unless its generated test samples have been reviewed; Gemini supplies the Kinyarwanda runtime path meanwhile.